# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MohamedRamadan164/FlyRank_ML_internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [20]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.


In [21]:
!{sys.executable} scripts/run_all.py



▶ Step 1/5 — Prepare features — clean the data, build the feature vector, define the label
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/data/processed/refresh_feature_vector.csv

▶ Step 2/5 — Baseline — a transparent hand-written rule to beat
Wrote baseline queue: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340

▶ Step 3/5 — Train — logistic regression, decision tree, random forest (client-holdout split)
Trained 3 models on 30,000 rows
Split strategy: client_holdout
Best model: random_forest
Wrote predictions: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/data/processed/model_predictions.csv
Wrote model results: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/outputs/model_results.json

▶ Step 4/5 — Evaluate — ranked refresh queu

In [22]:
import json
res = json.load(open("outputs/model_results.json"))

base = res["baseline"]["baseline_precision_at_50"]
rf   = res["models"]["random_forest"]["precision_at_50"]

print(f"Hand-written rule  Precision@50: {base:.3f}   (~{round(base*50)} of the top 50 right)")
print(f"Random forest      Precision@50: {rf:.3f}   (~{round(rf*50)} of the top 50 right)")
print(f"\nThe learned model roughly {rf/base:.1f}x the rule on this metric.")
print("Validation split used:", res["split_strategy"], "(pages from a client are never in both train and test)")

Hand-written rule  Precision@50: 0.240   (~12 of the top 50 right)
Random forest      Precision@50: 0.740   (~37 of the top 50 right)

The learned model roughly 3.1x the rule on this metric.
Validation split used: client_holdout (pages from a client are never in both train and test)


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Answer.** My lane is **Ranking Signal Analysis**: which safe, observable signals (position, freshness, CTR, engagement, word count, age) actually go with a page's traffic trend, and how strongly? I'm framing this as a **classification** task — separating pages whose trend is currently "down" from pages that are stable or growing — but I'm using the classifier as a **lens on the signals**, not as the finished product. What I want out of it is which signals carry real information and how strong that signal is, so a later lane (refresh scoring) knows which ones are worth building a ranked queue on. The code cell below shows the raw split I'd be classifying, so the framing isn't abstract.

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("rows, cols:", df.shape)
print()
print(df["trend_direction"].value_counts())


rows, cols: (30000, 44)

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*


**Answer.** The label is `is_declining_label = (trend_direction == "down")`. This is a **proxy, not an observed outcome**: `trend_direction` is a bucket the starter pipeline computes by comparing a page's last 30 days to its previous 30 days *inside the same snapshot* — it is not an independent future result. That means a model trained on it can only tell me "does this page currently look like the down-bucket," not "will this page decline next month." A stronger target for later weeks — once I move to the full warehouse — would be a genuine future-window label: features from a prior 90-day window predicting decline over the *next* 30 days. I'm flagging that limitation now on purpose rather than discovering it later.

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
base_rate = (df["trend_direction"] == "down").mean()
print(f"Share of pages currently labeled 'down': {base_rate:.1%}")
print(f"That's the baseline any classifier has to beat just by always guessing the majority.")


Share of pages currently labeled 'down': 54.2%
That's the baseline any classifier has to beat just by always guessing the majority.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Answer.** I'm using **ROC-AUC**. This is signal analysis, not a shipped review queue yet, so the question isn't "who reviews page 1 vs page 50" (that's Precision@K, for Lane 2) — it's the more basic question: *do these signals, combined, carry real information about the trend at all?* ROC-AUC answers exactly that: 0.5 means the signals are noise, 1.0 means perfect separation. "Good" for me means comfortably above 0.5 with a plain model, using only the safe columns. This repo's own pipeline already answers this on the same data: a logistic regression over these signals reaches ROC-AUC 0.700, a decision tree 0.742, a random forest 0.750 — all well above chance, which is the evidence that this lane has something to say before I spend more weeks on it.

In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import json

res = json.load(open("outputs/model_results.json"))
for name, key in [("logistic regression", "logistic_regression"), ("decision tree", "decision_tree"), ("random forest", "random_forest")]:
    auc = res["models"][key]["roc_auc"]
    print(f"{name:20s} ROC-AUC: {auc:.3f}")
print("\nAll comfortably above the 0.5 no-information line.")


logistic regression  ROC-AUC: 0.700
decision tree        ROC-AUC: 0.742
random forest        ROC-AUC: 0.750

All comfortably above the 0.5 no-information line.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Answer.** One row = **one content item (`content_id`) belonging to one client (`client_id`), summarized over its own trailing 90-day window.** It is not one row per day and not one row per client — each page gets exactly one row, with its own aggregated signals already computed (impressions, clicks, CTR, position, engagement, freshness, trend). The slice below keeps only the safe, observable columns my lane actually uses.

In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

safe_cols = [
    "content_id", "client_id", "content_type", "main_intent",
    "word_count", "content_age_days", "days_since_last_update",
    "impressions_90d", "clicks_90d", "ctr", "avg_position",
    "engagement_rate", "scroll_rate", "trend_direction", "trend_pct",
]

lane_df = df[safe_cols].copy()
print("unique content_id:", lane_df["content_id"].nunique(), "| unique client_id:", lane_df["client_id"].nunique())
lane_df.head()


unique content_id: 30000 | unique client_id: 32


,content_id,client_id,content_type,main_intent,word_count,content_age_days,days_since_last_update,impressions_90d,clicks_90d,ctr,avg_position,engagement_rate,scroll_rate,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,3221.0,187,20,3803,29,0.76,10.6,5.88,4.55,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,2481.0,445,25,15320,7,0.05,20.3,0.00,10.00,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,3515.0,141,20,12581,11,0.09,36.5,0.00,28.57,down,-60.9
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,NaN,463,22,11751,58,0.49,6.2,1.28,3.45,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,2803.0,263,14,19140,24,0.13,44.0,0.00,24.29,down,-34.7


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*
**Answer.** No single signal moves with the trend on its own — I checked, and every one-signal correlation with `trend_pct` is basically flat (all under 0.05 in size). If decline were explainable by one threshold ("pages older than X days decline" or "pages below position Y decline"), one of these would show up clearly. None does. Yet several models built from the *combination* of these same weak signals reach ROC-AUC 0.70–0.75 (section 3) and a hand-written rule baseline only hits Precision@50 = 0.240 (~12 of the top 50 right) versus 0.740 (~37 of 50) for a random forest, per `outputs/model_report.md`. That gap is the evidence: the signals only become informative once something can weigh and combine them together, which is exactly what an if-statement can't do and a model can.

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
for col in ["days_since_last_update", "avg_position", "ctr", "engagement_rate", "word_count"]:
    c = df[[col, "trend_pct"]].dropna()
    print(f"{col:22s} corr with trend_pct: {c[col].corr(c['trend_pct']):+.3f}")

print("\nEach signal alone barely correlates with the trend — but combined models do far better (section 3).")


days_since_last_update corr with trend_pct: -0.014
avg_position           corr with trend_pct: +0.047
ctr                    corr with trend_pct: +0.008
engagement_rate        corr with trend_pct: +0.008
word_count             corr with trend_pct: -0.009

Each signal alone barely correlates with the trend — but combined models do far better (section 3).


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.